### Settings

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

from pathlib import Path
BASE_DIR = Path().resolve().parent   
DATA_DIR = BASE_DIR / "data"

In [2]:
from IPython.display import display, HTML

display(HTML("""
<style>
.output pre {
    white-space: pre-wrap;
}
</style>
"""))

In [3]:
def show_full(df):
    """
    Fully display a DataFrame without any truncation.
    Works for any df[...] slice.
    """
    import pandas as pd
    from IPython.display import display

    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None,
        "display.max_columns", None,
        "display.width", None
    ):
        display(df)

### Read data 

In [ ]:
df = pd.read_parquet('https://storage.googleapis.com/msca-bdp-data-open/news_final_project/news_final_project.parquet', engine='pyarrow')
print(df.shape)
cols = ['url', 'date', 'language', 'title', 'text']
print(cols)

In [ ]:
# Show a short preview to understand what the raw text looks like
ex = df[['title', 'text']].dropna().sample(5, random_state=42)
for i, (idx, row) in enumerate(ex.iterrows(), 1):
    print("\n" + "="*80)
    print(f"[Example {i}]")
    print(f"Title: {row['title']}")
    print(f"Text preview (first 500 chars):\n{row['text'][:500]}")

In [ ]:
# Analyze text length distribution
text_lengths = df['text'].dropna().astype(str).apply(len)
print("Text length distribution (characters):")
print(text_lengths.describe())

plt.figure(figsize=(12, 2))
text_lengths.plot(kind='box', vert=False)

p = 0.01
q = text_lengths.quantile(p)
print(f"{p*100}% percentile length:", q)

In [ ]:
# Analyze title length distribution
print(df['title'].dropna().astype(str).apply(len).describe())
df['title'].dropna().astype(str).apply(len).quantile(0.99)

In [ ]:
df['text'].isna().sum()

In [ ]:
df['language'].value_counts()

### The lightest and structure-preserving normalization

In [ ]:
def normalize_whitespace(text: str) -> str:
    """
    Light normalization that preserves structure (paragraph boundaries).
    """

    # 1) Normalize all newline styles to '\n'
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # 2) Collapse multiple spaces or tabs into a single space
    text = re.sub(r"[ \t]+", " ", text)

    # 3) Collapse excessive blank lines (3 or more) into double newline
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 4) Trim leading and trailing whitespace
    text = text.strip()

    return text

In [ ]:
df0 = df.copy()
df0["text_norm0"] = df["text"].astype(str).map(normalize_whitespace)
df0["text_norm0_lengths"] = df0["text_norm0"].astype(str).apply(len)

In [ ]:
# Identify rows where normalization changed the text
mask_changed = df0["text"] != df0["text_norm0"]

print("Changed ratio:", mask_changed.mean())
print("Changed count:", mask_changed.sum())

### Remove docs where length <= 1000
I read through those text, they are mostly just title + boilerplate. That text ≈ title provides almost no info from my perspective.

Even if some normal text are wrongly deleted, it won't affect a lot due to the proportion.

In [ ]:
df1 = df0.copy().query("text_norm0_lengths > 1000")

print("Before:", len(df0), "After:", len(df1))

In [ ]:
# Check docs where title = text
from difflib import SequenceMatcher

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

# --- prepare strings ---
df1["title_clean"] = df1["title"].fillna("").astype(str)
df1["text_clean"]  = df1["text_norm0"].fillna("").astype(str)

# --- restrict to short docs only (speed gate) ---
short_mask = df1["text_clean"].str.len() < 5000

# pre-allocate sim_ratio with NaN for all rows
df1["sim_ratio"] = np.nan

# compute similarity only for short docs
titles = df1.loc[short_mask, "title_clean"].tolist()
texts  = df1.loc[short_mask, "text_clean"].tolist()

df1.loc[short_mask, "sim_ratio"] = [similarity(a, b) for a, b in zip(titles, texts)]

# len_diff is cheap, you can compute for all rows
df1["len_diff"] = (df1["title_clean"].str.len() - df1["text_clean"].str.len()).abs()

# --- define duplicate mask ---
dup_mask = (
    (df1["title_clean"] == df1["text_clean"]) |
    (short_mask & (
        ((df1["len_diff"] <= 20) & (df1["sim_ratio"] > 0.85)) |
        (df1["sim_ratio"] > 0.9)
    ))
)

dup_rows = df1.loc[dup_mask]

print("Near-duplicate rows:", len(dup_rows))
dup_rows.head(2)

### Remove HTML artifacts 

In [ ]:
df2 = df1[~dup_mask].copy()[cols + ["text_norm0", "text_norm0_lengths"]]


In [ ]:
# --- HTML artifact signals ---
tag_re    = re.compile(r"<[^>]+>")        # matches <p>...</p>, <br>, <div ...>, etc.
entity_re = re.compile(r"&[a-zA-Z]+;|&#\d+;")  # matches &amp; &nbsp; &#39; etc.

# --- Count number of HTML tags per article ---
df_sample = df2["text_norm0"].dropna().astype(str).sample(5000, random_state=42).copy()

df_sample_tag_count = df_sample.apply(lambda x: len(tag_re.findall(x)))
print("== tag count ==")
print(df_sample_tag_count.describe())

df_sample_entity_count = df_sample.apply(lambda x: len(entity_re.findall(x)))
print("\n== entity count ==")
print(df_sample_entity_count.describe())

Profiling shows that over 75% of documents contain no HTML tags or entities. The average HTML density is near zero, indicating that the dataset is largely pre-cleaned. Therefore, we apply a uniform HTML stripping procedure for consistency.

> Regex is the "deletion mode" at the string level

> BeautifulSoup is a structure-level "HTML parsing" service

In [ ]:
from bs4 import BeautifulSoup
import html

def clean_html_fast(text):
    if not tag_re.search(text) and not entity_re.search(text):
        return text  # no HTML, skip expensive parsing

    text = html.unescape(text)
    soup = BeautifulSoup(text, "lxml")
    return soup.get_text(" ", strip=True)

# Apply to full dataset
df2["text_clean_v1"] = df2["text_norm0"].astype(str).apply(clean_html_fast)

In [ ]:
df3 = df2.copy()

save_path_v1 = DATA_DIR / "temp" / "text_clean_v1.parquet"
df3.to_parquet(save_path_v1, index=False)

### Irrelevant crawl artifacts

In [ ]:
save_path_v1 = DATA_DIR / "temp" / "text_clean_v1.parquet"
df3 = pd.read_parquet(save_path_v1)
df3.head(2)

#### Quick boilerplate filter on overall level
In this part, I as ChatGPT to generate a function that can return a bp_score, which is a 0–1 “junk likelihood” score for a crawled page-text. It’s computed from simple, interpretable signals—no ML—then combined with weights.

It’s computed from two parts: basic html signals (url, short_line, repetitive lines ) and boilerplate (5 boilerplate categories).

Html signals is calculated in ratio, while boilerplate is 0–1 component scores if the hit counts reach a soft/hard threshold.

> Navigation/menu/account terms (e.g., “Sign in”, “Markets”, “Privacy policy”)

> Ads/marketing terms (e.g., “ADVERTISEMENT”, “sponsored”, ad-tech names)

> Comments/forms terms (e.g., “Leave a reply”, “This field is required”)

> Related/trending modules (e.g., “Most popular”, “Related stories”)

>  Directory/listing pages (e.g., “Products”, “Upvote”, “Launch archive”)

Finally the weighted sum → final bp_score

The function is more complex because it try to store the main components and give suggestions accordingly. But considering the time limit, I will just go with simply give a score and decide the threshold myself.


In [ ]:
import re
from collections import Counter
from typing import Dict, Any, List, Tuple

# -----------------------------
# 0) Regex building blocks
# -----------------------------
URL_RE = re.compile(r"https?://\S+", re.IGNORECASE)
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")

def _normalize_newlines(text: str) -> str:
    # handle literal "\n" from crawling dumps
    if "\\n" in text and "\n" not in text:
        text = text.replace("\\n", "\n")
    return text

def _safe_str(x) -> str:
    if x is None:
        return ""
    try:
        return str(x)
    except Exception:
        return ""

def _match_count(patterns: List[re.Pattern], text: str) -> Tuple[int, List[str]]:
    hits = []
    for p in patterns:
        for m in p.finditer(text):
            hits.append(m.group(0))
    return len(hits), hits[:15]  # cap evidence list

def _line_stats(text: str) -> Dict[str, float]:
    lines = [ln.strip() for ln in text.splitlines()]
    lines = [ln for ln in lines if ln]  # non-empty
    if not lines:
        return {"n_lines": 0, "avg_line_len": 0.0, "short_line_ratio": 0.0, "unique_line_ratio": 0.0}
    lens = [len(ln) for ln in lines]
    short_line_ratio = sum(l <= 25 for l in lens) / len(lens)
    unique_line_ratio = len(set(lines)) / len(lines)
    return {
        "n_lines": float(len(lines)),
        "avg_line_len": float(sum(lens) / len(lens)),
        "short_line_ratio": float(short_line_ratio),
        "unique_line_ratio": float(unique_line_ratio),
    }

def _tokenize_simple(text: str) -> List[str]:
    # cheap tokenization for keyword density
    return re.findall(r"[A-Za-z']+", text.lower())

def _keyword_density(tokens: List[str], keyword_set: set) -> float:
    if not tokens:
        return 0.0
    c = sum(1 for t in tokens if t in keyword_set)
    return c / len(tokens)

def _clamp01(x: float) -> float:
    return max(0.0, min(1.0, x))

def _score_from_count(count: int, soft: int, hard: int) -> float:
    """
    Map a count -> [0,1] by soft/hard thresholds.
    <=soft -> 0..~0.4, >=hard -> 1.0
    """
    if count <= 0:
        return 0.0
    if count >= hard:
        return 1.0
    if count <= soft:
        return 0.4 * (count / soft)
    # between soft and hard: ramp 0.4 -> 1.0
    return 0.4 + 0.6 * ((count - soft) / (hard - soft))


# -----------------------------
# 1) Patterns for 5 pollution categories
# -----------------------------

# (1) Navigation / menu / account / edition blocks
NAV_PATTERNS = [
    re.compile(r"\b(sign in|log in|logout|my account|settings)\b", re.IGNORECASE),
    re.compile(r"\b(home|about|contact|careers|faq|apps)\b", re.IGNORECASE),
    re.compile(r"\b(edition|international|arabic|español|espanol)\b", re.IGNORECASE),
    re.compile(r"\b(newsletters?|e-?paper|subscribe|sign up)\b", re.IGNORECASE),
    re.compile(r"\b(topics you follow|follow us)\b", re.IGNORECASE),
    re.compile(r"\b(markets|tech|media|calculators|videos|live tv)\b", re.IGNORECASE),
    re.compile(r"\b(privacy policy|terms|disclaimer|cookie|gdpr)\b", re.IGNORECASE),
    re.compile(r"\b(menu|toggle|×|close icon)\b", re.IGNORECASE),
]

# (2) Ads / marketing injection
AD_PATTERNS = [
    re.compile(r"\b(advertisement|ADVERTISEMENT|sponsored|promoted)\b", re.IGNORECASE),
    re.compile(r"\b(ad feedback|ad choices|ad never loaded|ad prevented)\b", re.IGNORECASE),
    re.compile(r"\b(1xbet|parimatch|mostbet|stake|betting|signup bonus)\b", re.IGNORECASE),
    re.compile(r"\b(microsoft clarity|doubleclick|taboola|outbrain)\b", re.IGNORECASE),
]

# (3) Comments / forms / input validation
COMMENT_PATTERNS = [
    re.compile(r"\b(leave a reply|cancel reply|comments?)\b", re.IGNORECASE),
    re.compile(r"\b(your email address will not be published|required fields)\b", re.IGNORECASE),
    re.compile(r"\b(this field is required|please enter a valid email)\b", re.IGNORECASE),
    re.compile(r"\b(save my name, email, and website)\b", re.IGNORECASE),
    re.compile(r"\b(submit|thank you|close)\b", re.IGNORECASE),
]

# (4) Related / popular / trending / sidebar modules
RELATED_PATTERNS = [
    re.compile(r"\b(most popular|top news|latest news|trending)\b", re.IGNORECASE),
    re.compile(r"\b(related stories|you might also like|read more|explainers?)\b", re.IGNORECASE),
    re.compile(r"\b(article continues below)\b", re.IGNORECASE),
    re.compile(r"\b(recent posts|categories)\b", re.IGNORECASE),
]

# (5) Directory/listing/non-article page signals (Product Hunt / tag pages / archive)
DIRECTORY_PATTERNS = [
    re.compile(r"\b(products|collections|marketplace|launch archive|coming soon)\b", re.IGNORECASE),
    re.compile(r"\b(upvote|comments?\s*\d+|day rank|week rank|featured on)\b", re.IGNORECASE),
    re.compile(r"\b(browse products|topics|jobs|post a job|advertise)\b", re.IGNORECASE),
    re.compile(r"\b(story's credibility|about this launch|makers of)\b", re.IGNORECASE),
]

# Optional: Context-like “Viewing:” rail
VIEWING_PATTERNS = [
    re.compile(r"\bViewing:\b", re.IGNORECASE),
    re.compile(r"\bShare\b.*\bTweet\b.*\bPost\b.*\bEmail\b", re.IGNORECASE | re.DOTALL),
]

# Keyword sets for density checks
NAV_KEYWORDS = {
    "home","about","contact","careers","apps","faq","edition","international","markets","tech","media",
    "videos","subscribe","newsletter","login","logout","signin","signup","privacy","terms","cookie"
}
AD_KEYWORDS = {"advertisement","sponsored","promoted","betting","bonus","ad"}
COMMENT_KEYWORDS = {"comment","comments","reply","submit","required","email"}
RELATED_KEYWORDS = {"related","popular","trending","latest","stories","read","more","explained","explainer"}
DIRECTORY_KEYWORDS = {"products","collections","marketplace","launch","upvote","featured","rank","makers","archive"}


# -----------------------------
# 2) Main scoring function
# -----------------------------
def boilerplate_score(text: str, return_clean_suggestion: bool = True) -> Dict[str, Any]:
    """
    Score how likely `text` is web-crawling boilerplate / non-article junk.

    Covers 5 pollution categories:
      1) navigation/menu blocks
      2) ads/marketing blocks
      3) comments/forms blocks
      4) related/popular/sidebar blocks
      5) directory/listing pages

    Returns:
      {
        "score": 0..1,
        "components": {...},
        "evidence": {...},
        "features": {...},
        "decision": {...}
      }
    """
    raw = _safe_str(text)
    t = _normalize_newlines(raw).strip()

    if not t:
        return {
            "score": 1.0,
            "components": {k: 1.0 for k in ["nav","ads","comments","related","directory"]},
            "evidence": {"reason": ["empty text"]},
            "features": {"len": 0},
            "decision": {"label": "boilerplate", "confidence": "high", "suggest": "drop"},
        }

    # Basic features
    n_chars = len(t)
    urls = URL_RE.findall(t)
    url_chars = sum(len(u) for u in urls)
    url_ratio = (url_chars / n_chars) if n_chars else 0.0
    email_cnt = len(EMAIL_RE.findall(t))
    stats = _line_stats(t)
    tokens = _tokenize_simple(t)

    # Pattern hit counts + evidence snippets
    nav_cnt, nav_hits = _match_count(NAV_PATTERNS, t)
    ad_cnt, ad_hits = _match_count(AD_PATTERNS, t)
    com_cnt, com_hits = _match_count(COMMENT_PATTERNS, t)
    rel_cnt, rel_hits = _match_count(RELATED_PATTERNS, t)
    dir_cnt, dir_hits = _match_count(DIRECTORY_PATTERNS, t)
    view_cnt, view_hits = _match_count(VIEWING_PATTERNS, t)

    # Density signals (helps with "menu word soup" pages)
    nav_density = _keyword_density(tokens, NAV_KEYWORDS)
    ad_density = _keyword_density(tokens, AD_KEYWORDS)
    com_density = _keyword_density(tokens, COMMENT_KEYWORDS)
    rel_density = _keyword_density(tokens, RELATED_KEYWORDS)
    dir_density = _keyword_density(tokens, DIRECTORY_KEYWORDS)

    # Component scoring (each -> [0,1])
    # counts thresholds are tuned for typical news-crawl dumps; tweak if needed
    s_nav = _score_from_count(nav_cnt, soft=3, hard=12)
    s_ads = _score_from_count(ad_cnt, soft=1, hard=5)
    s_com = _score_from_count(com_cnt, soft=1, hard=6)
    s_rel = _score_from_count(rel_cnt, soft=1, hard=6)
    s_dir = _score_from_count(dir_cnt, soft=1, hard=6)

    # Add density & structural boosters (clamped)
    # URL ratio: in real article usually tiny; if >2% it's suspicious; >8% very suspicious
    s_url = _clamp01((url_ratio - 0.02) / 0.06)  # 0 at 2%, 1 at 8%
    # Many short lines suggests menus/rails
    s_shortlines = _clamp01((stats["short_line_ratio"] - 0.25) / 0.45)  # 0 at 25%, 1 at 70%
    # very low unique-line ratio can indicate repeated rails; but also legitimate repeated lines, so mild
    s_repetition = _clamp01((0.85 - stats["unique_line_ratio"]) / 0.35)

    # density boosters (small weights)
    s_nav = _clamp01(s_nav + 0.8 * nav_density)
    s_ads = _clamp01(s_ads + 1.2 * ad_density)
    s_com = _clamp01(s_com + 1.0 * com_density)
    s_rel = _clamp01(s_rel + 0.8 * rel_density)
    s_dir = _clamp01(s_dir + 1.0 * dir_density)

    # Viewing/share rail is a strong indicator; treat as additive kicker
    s_view = _clamp01(view_cnt / 2.0)  # 0, 0.5, 1

    # Weighted total score (interpretable weights)
    # - navigation + directory are most common "non-article" predictors
    # - url_ratio + shortlines capture DOM dumps
    total = (
        0.22 * s_nav +
        0.18 * s_dir +
        0.16 * s_ads +
        0.14 * s_com +
        0.10 * s_rel +
        0.10 * s_url +
        0.06 * s_shortlines +
        0.04 * s_repetition
    )
    total = _clamp01(total + 0.10 * s_view)

    # Decision policy (simple, adjustable)
    # - high score => likely boilerplate page, drop
    # - mid score => keep but try cleaning rules
    # - low score => treat as content
    if total >= 0.70:
        label, conf, suggest = "boilerplate", "high", "drop_or_recrawl"
    elif total >= 0.45:
        label, conf, suggest = "mixed", "medium", "clean_then_keep"
    else:
        label, conf, suggest = "content", "medium" if total >= 0.25 else "high", "keep"

    out = {
        "score": float(total),
        "components": {
            "nav": float(s_nav),
            "ads": float(s_ads),
            "comments": float(s_com),
            "related": float(s_rel),
            "directory": float(s_dir),
            "url": float(s_url),
            "shortlines": float(s_shortlines),
            "repetition": float(s_repetition),
            "viewing_rail": float(s_view),
        },
        "evidence": {
            "nav_hits": nav_hits,
            "ad_hits": ad_hits,
            "comment_hits": com_hits,
            "related_hits": rel_hits,
            "directory_hits": dir_hits,
            "viewing_hits": view_hits,
        },
        "features": {
            "len_chars": int(n_chars),
            "n_urls": int(len(urls)),
            "url_ratio": float(url_ratio),
            "email_cnt": int(email_cnt),
            **stats,
            "nav_density": float(nav_density),
            "ad_density": float(ad_density),
            "comment_density": float(com_density),
            "related_density": float(rel_density),
            "directory_density": float(dir_density),
            "pattern_counts": {
                "nav": int(nav_cnt),
                "ads": int(ad_cnt),
                "comments": int(com_cnt),
                "related": int(rel_cnt),
                "directory": int(dir_cnt),
                "viewing": int(view_cnt),
            },
        },
        "decision": {"label": label, "confidence": conf, "suggest": suggest},
    }

    # Optional: a very lightweight cleaning suggestion (NOT performing cleaning here)
    if return_clean_suggestion:
        out["decision"]["recommended_actions"] = []
        if s_ads >= 0.6:
            out["decision"]["recommended_actions"].append("strip_ad_blocks")
        if s_nav >= 0.6:
            out["decision"]["recommended_actions"].append("strip_nav_footer")
        if s_com >= 0.6:
            out["decision"]["recommended_actions"].append("strip_comments_forms")
        if s_rel >= 0.6:
            out["decision"]["recommended_actions"].append("strip_related_modules")
        if s_dir >= 0.6:
            out["decision"]["recommended_actions"].append("drop_directory_pages")
        if s_url >= 0.6:
            out["decision"]["recommended_actions"].append("filter_high_url_density")
        if s_view >= 0.5:
            out["decision"]["recommended_actions"].append("strip_viewing_share_rail")

    return out

def boilerplate_score_only(text):
    return boilerplate_score(text, return_clean_suggestion=False)["score"]

In [ ]:
# # 1) get score + label
# tmp = df1["text_norm0"].fillna("").astype(str).apply(boilerplate_score)

# df1["bp_score"] = tmp.apply(lambda d: d["score"])
# df1["bp_label"] = tmp.apply(lambda d: d["decision"]["label"])
# df1["bp_suggest"] = tmp.apply(lambda d: d["decision"]["suggest"])

# # 2)  top junk
# df1.sort_values("bp_score", ascending=False)[
#     ["url", "title_clean", "text_norm0_lengths", "bp_score", "bp_label", "bp_suggest"]
# ].head(20)

# only get score for faster processing 
df3["bp_score"] = df3["text_clean_v1"].apply(boilerplate_score_only)

In [ ]:
save_path_bpscore = DATA_DIR / "temp" / "text_bp_score.parquet"
df3.to_parquet(save_path_bpscore, index=False)

In [4]:
save_path_bpscore = DATA_DIR / "temp" / "text_bp_score.parquet"
df3 = pd.read_parquet(save_path_bpscore)
df3.head(2)

,url,date,language,title,text,text_norm0,text_norm0_lengths,text_clean_v1,bp_score
0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...",3500,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",0.090024
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,\n\nThis AI video of gymnastics might be the f...,This AI video of gymnastics might be the freak...,4924,This AI video of gymnastics might be the freak...,0.459821


In [5]:
check_boilerplate = df3[(df3["bp_score"] >= 0.6) ][["text_norm0", "bp_score",'text_norm0_lengths']].sort_values("bp_score", ascending=True)
len(check_boilerplate)

23127

Docs with bp_score >= 0.6 has more than half of the paragraphs are boilerplates.

In [6]:
# I check the results according to the score and decide the threshold to be 0.6
df3["bp_label"] = pd.cut(
    df3["bp_score"],
    bins=[-0.01, 0.4, 0.6, 1.0],
    labels=["content", "mixed", "boilerplate"]
)
df3[df3["bp_label"] == "boilerplate"].__len__() # check porpotion

23127

In [7]:
# Quick qualitative check of "boilerplate" rows
def print_full_rows(df, text_col="text_clean_v1", extra_cols=None, n=5, random_state=42):
    sample_df = df.sample(min(n, len(df)), random_state=random_state)

    for i, (idx, row) in enumerate(sample_df.iterrows(), 1):
        print("\n" + "="*120)
        print(f"[Row {i}] Index: {idx}")
        print("="*120)

        if extra_cols:
            for c in extra_cols:
                print(f"{c}: {row[c]}")
            print("-"*120)

        print(row[text_col])
        print("\n" + "="*120)

print_full_rows(
    df3[df3["bp_label"] == "boilerplate"],
    text_col="text_clean_v1",
    extra_cols=["bp_score"],
    n=5
)


[Row 1] Index: 167071
bp_score: 0.707617468993063
------------------------------------------------------------------------------------------------------------------------
Artificial intelligence: Suspected Chinese operatives using AI generated images to spread disinformation among US voters, Microsoft says | CNN Politics

CNN values your feedback

 1. How relevant is this ad to you?
 

 2. Did you encounter any technical issues?
 
 Video player was slow to load content
 
 Video content never loaded
 
 Ad froze or did not finish loading
 
 Video content did not start after ad
 
 Audio on ad was too loud
 
 Other issues
 
 Ad never loaded
 
 Ad prevented/slowed the page from loading
 
 Content moved around while ad loaded
 
 Ad was repetitive to ads I've seen previously
 
 Other issues
 
 Cancel
 

 Submit
 
Thank You!

 Your effort and contribution in providing this feedback is much
 appreciated.
 

 Close
 
Ad Feedback

Close icon
 
 SCOTUS
 
 
 Congress
 
 
 Facts First
 
 
 2024 Ele

In [8]:
df4 = df3[df3["bp_label"] != "boilerplate"].copy()
print("Before:", len(df3), "After:", len(df4))

Before: 191666 After: 168539


#### Sign-in specific

In [9]:
signin_re = re.compile(
    r"\b("
    r"sign\s*in|log\s*in|login|"
    r"sign\s*out|log\s*out|logout|"   
    r"welcome!\s*log\s*into\s*your\s*account|"
    r"forgot\s+your\s+password|password\s+recovery|recover\s+your\s+password|"
    r"your\s+username|your\s+password|"
    r"my\s+account|create\s+account|register|sign\s*up"
    r")\b",
    flags=re.IGNORECASE
)

# login type button words, they are treated differently in case of mismatch in long text with a part of the phrases
def has_login_phrase(s: str) -> bool:
    s_lower = s.lower()
    return any(
        phrase in s_lower
        for phrase in [
            "log in",
            "login",
            "sign in",
            "sign up",
            "create account",
            "my account",
            "register",
            "subscribe",
            "logout",
            "log out",    
            "sign out",   
        ]
    )

# Heuristic to identify lines that look like login/signup buttons or menu items
def is_button_like_line(ln: str) -> bool:
    s = ln.strip()
    if not s:
        return False

    if len(s) > 32:
        return False

    letters = [ch for ch in s if ch.isalpha()]
    if letters:
        upper_ratio = sum(ch.isupper() for ch in letters) / len(letters)
    else:
        upper_ratio = 0.0

    toks = re.findall(r"[A-Za-z]+", s.lower())
    if not toks:
        return False

    short_word_ratio = sum(len(t) <= 6 for t in toks) / len(toks)

    has_login_word = has_login_phrase(s)

    return has_login_word and (upper_ratio >= 0.6 or short_word_ratio >= 0.8)


MAX_DEL_LINE_LEN = 100  # in case of removing a whole line that contains a login phrase

def remove_signin_lines(text, max_len=MAX_DEL_LINE_LEN):
    if not text:
        return text

    text = str(text)
    if "\\n" in text and "\n" not in text:
        text = text.replace("\\n", "\n")

    cleaned_lines = []
    for ln in text.splitlines():
        s = ln.strip()

        if len(s) <= max_len and signin_re.search(s):
            continue
        if len(s) <= max_len and is_button_like_line(s):
            continue

        cleaned_lines.append(ln)

    return "\n".join(cleaned_lines).strip()

In [10]:
# print some examples of removed lines to verify
def extract_removed_signin_like_lines(text, max_len=MAX_DEL_LINE_LEN):
    if not text:
        return []

    text = str(text)
    if "\\n" in text and "\n" not in text:
        text = text.replace("\\n", "\n")

    removed = []
    for ln in text.splitlines():
        s = ln.strip()
        if len(s) <= max_len and (signin_re.search(s) or is_button_like_line(s)):
            removed.append(s)
    return removed

df_with = df4[df4["text_clean_v1"].apply(lambda x: len(extract_removed_signin_like_lines(x)) > 0)]

for idx in df_with.sample(5, random_state=42).index:
    print("\n" + "="*100)
    print("Index:", idx)
    for ln in extract_removed_signin_like_lines(df4.loc[idx, "text_clean_v1"]):
        print("REMOVED:", ln)


Index: 136040
REMOVED: Login
REMOVED: Register Now

Index: 84496
REMOVED: Log in
REMOVED: Sign In
REMOVED: Sign up for Smart Investing to get the latest news, strategies and tips to help you invest smarter.

Index: 155319
REMOVED: Sign in
REMOVED: My Account
REMOVED: REGISTER FOR FREE
REMOVED: Sign In
REMOVED: Sign out
REMOVED: My Account
REMOVED: Sign in

Index: 132681
REMOVED: Log in
REMOVED: Log in
REMOVED: Sign up for notifications from Insider! Stay up to date with what you want to know.

Index: 50840
REMOVED: Login
REMOVED: Register Now
REMOVED: Register Now
REMOVED: Register Now
REMOVED: Register Now
REMOVED: Register Now
REMOVED: Register Now
REMOVED: Register Now


In [11]:
df4["text_clean_v2"] = df4["text_clean_v1"].apply(remove_signin_lines)

In [12]:
df5 = df4.copy().drop(columns=["text_norm0_lengths"])
df5['text_v2_length'] = df5['text_clean_v2'].astype(str).apply(len)
df5.columns

Index(['url', 'date', 'language', 'title', 'text', 'text_norm0',
       'text_clean_v1', 'bp_score', 'bp_label', 'text_clean_v2',
       'text_v2_length'],
      dtype='object')

#### Cookies and privacy polices - Specific

In [13]:
# Must mention cookie(s)
cookie_trigger_re = re.compile(r"\bcookies?\b", re.IGNORECASE)

# Consent / policy / action language commonly co-occurring with cookie banners
cookie_context_re = re.compile(
    r"\b("
    r"we\s+use\s+cookies|"
    r"this\s+website\s+uses\s+cookies|"
    r"by\s+continuing\s+to\s+use|"
    r"you\s+agree\s+to\s+our\s+use\s+of\s+cookies|"
    r"cookie\s+settings|cookie\s+policy|"
    r"privacy\s+policy|terms(?:\s+of\s+use)?|terms\s*&\s*conditions|"
    r"consent|gdpr|"
    r"learn\s+more|accept|i\s+agree|reject|manage"
    r")\b",
    flags=re.IGNORECASE
)

MAX_COOKIE_LINE_LEN = 600  # cookie banners can be longer than sign-in buttons
def remove_cookie_lines(text, max_len=MAX_COOKIE_LINE_LEN):
    
    text = str(text)
    if "\\n" in text and "\n" not in text:
        text = text.replace("\\n", "\n")

    kept = []
    for ln in text.splitlines():
        s = ln.strip()
        if s and len(s) <= max_len and cookie_trigger_re.search(s) and cookie_context_re.search(s):
            continue
        kept.append(ln)

    return "\n".join(kept).strip()

In [14]:
def extract_removed_cookie_lines(text, max_len=MAX_COOKIE_LINE_LEN):

    text = str(text)
    if "\\n" in text and "\n" not in text:
        text = text.replace("\\n", "\n")

    removed = []
    for ln in text.splitlines():
        s = ln.strip()
        if not s:
            continue

        if len(s) > max_len:
            continue

        if cookie_trigger_re.search(s) and cookie_context_re.search(s):
            removed.append(s)

    return removed

df_cookie_with = df5[
    df5["text_clean_v2"].apply(lambda x: len(extract_removed_cookie_lines(x)) > 0)
]

for idx in df_cookie_with.sample(5, random_state=42).index:
    print("\n" + "="*100)
    print("Index:", idx)
    for ln in extract_removed_cookie_lines(df5.loc[idx, "text_clean_v2"]):
        print("REMOVED:", ln)


Index: 21025
REMOVED: We use cookies for analytics, personalized content and ads. By continuing to browse this site, you agree to this use.

Index: 80792
REMOVED: Mondaq uses cookies on this website. By using our website you agree to our use of cookies as set out in our Privacy Policy.

Index: 155376
REMOVED: Cookie Settings

Index: 114242
REMOVED: This website uses cookies to improve your experience. We'll assume you're ok with this, but you can opt-out if you wish. Accept Read More

Index: 189839
REMOVED: Cookie settings


In [15]:
df5["text_clean_v3"] = df4["text_clean_v2"].apply(remove_cookie_lines)

In [16]:
df6 = df5.copy().drop(columns=['text_v2_length'])
df6['text_v3_length'] = df6['text_clean_v3'].astype(str).apply(len)
df6.columns

Index(['url', 'date', 'language', 'title', 'text', 'text_norm0',
       'text_clean_v1', 'bp_score', 'bp_label', 'text_clean_v2',
       'text_clean_v3', 'text_v3_length'],
      dtype='object')

#### Short runs from blocks

During the cleaning process, I observed that many documents contained repeated clusters of very short lines grouped within \n\n-separated blocks. These clusters were typically navigation menus, footer links, subscription prompts, or other structural artifacts — likely residual HTML elements that were flattened into text during crawling.

Because these “short line runs” were highly patterned and disproportionately located at the beginning or end of documents, I implemented a rule-based function to detect and remove consecutive short-line sequences within paragraph blocks (split by \n\n). The removal is constrained to edge regions of the document and includes semantic guards to avoid deleting topic-relevant content.

**Code logic**
- Split the text into lines with splitlines().
A run is consecutive non-empty lines where len(line.strip()) <= max_chars.

- Edge-only deletion (top/bottom 30%): Only delete runs whose start line is in the top region or in the bottom region. So if a menu rail appears in the middle (e.g., a legitimate bullet list or dialogue), it won’t be removed.

- Semantic protection layer (AI-related) is add in order not to wrongly remove anything meaningful

In [17]:
PROTECTED_KEYWORDS = {
    "ai","artificial intelligence","tech","technology","cloud",
    "ml","machine learning","data science","deep learning","learning","data"
    "model","models","software","programming","developer","developers",
    "llm","gpt",
    "impact","replace","career","job","employment","careers","opportunities","opportunity",
    "forecaset","trends","analysis","research","report","insights","insight",
}

def normalize_newlines_loose(t: str) -> str:
    t = "" if t is None else str(t)

    # 1) Convert escaped newlines first (works even if real \n already exists)
    t = t.replace("\\r\\n", "\n").replace("\\n", "\n").replace("\\r", "\n")

    # 2) Normalize real Windows/Mac newlines too
    t = t.replace("\r\n", "\n").replace("\r", "\n")

    # 3) Collapse excessive blank lines to keep block splitting stable
    t = re.sub(r"\n{3,}", "\n\n", t)

    return t


In [18]:
def extract_short_line_runs_from_blocks(
    text,
    max_chars: int = 50,
    min_run: int = 1,
    edge_ratio: float = 0.30,
    protected_keywords=PROTECTED_KEYWORDS,
):
    """
    Extract consecutive short-line runs (len<=max_chars) that lie fully within
    the first/last edge_ratio of the FULL text.

    Returns a list of dict runs:
      {side, block_i, n_lines, run_start, run_end, lines}
    """
    # normalize 
    t = normalize_newlines_loose(text)  

    L = len(t)
    if L == 0:
        return []

    left_end  = int(edge_ratio * L)
    right_beg = int((1.0 - edge_ratio) * L)

    sep = "\n\n"
    sep_len = len(sep)

    runs = []
    pos = 0  # running cursor in FULL TEXT (exact)

    # split blocks by \n\n WITHOUT strip/filter, so spans are exact
    parts = t.split(sep)
    for bi, part in enumerate(parts):
        block_start = pos
        block_end   = block_start + len(part)

        # advance pos to next block start (include separator length)
        pos = block_end + sep_len

        # we only *process* non-empty content, but we did NOT drop it for span accuracy
        if not part.strip():
            continue

        # within-block split by \n, keep raw line spans by accumulating lengths
        # (splitlines(True) keeps the newline characters, so span math stays exact)
        raw_lines = part.splitlines(True)

        # build (line_text_stripped, line_start_in_full, line_end_in_full)
        line_spans = []
        line_pos = block_start
        for rl in raw_lines:
            s = rl.strip()
            if s:  # ignore blank lines for run formation
                line_spans.append((s, line_pos, line_pos + len(rl)))
            line_pos += len(rl)

        # find consecutive short lines
        i = 0
        while i < len(line_spans):
            s_i, start_i, _ = line_spans[i]
            if len(s_i) <= max_chars:
                j = i
                while j < len(line_spans) and len(line_spans[j][0]) <= max_chars:
                    j += 1

                if (j - i) >= min_run:
                    run_lines = [line_spans[k][0] for k in range(i, j)]
                    run_text = " ".join(run_lines).lower()

                    # semantic guard: if run contains topic keywords, do NOT mark as deletable
                    if protected_keywords and any(pk in run_text for pk in protected_keywords):
                        i = j
                        continue

                    run_start = line_spans[i][1]
                    run_end   = line_spans[j - 1][2]  # end of last raw line (incl newline)

                    # edge gating: entire run must be within left edge OR within right edge
                    in_left  = (run_end <= left_end)
                    in_right = (run_start >= right_beg)

                    if in_left or in_right:
                        runs.append({
                            "side": "top30%" if in_left else "bottom30%",
                            "block_i": bi,
                            "n_lines": (j - i),
                            "run_start": run_start,
                            "run_end": run_end,
                            "lines": run_lines,
                        })

                i = j
            else:
                i += 1

    return runs

In [19]:
def print_short_runs_for_two_texts(
    text1,
    max_chars=50,
    min_run=1,
    edge_ratio=0.30
):
    texts = [text1]

    for i, text in enumerate(texts, 1):
        runs = extract_short_line_runs_from_blocks(
            text,
            max_chars=max_chars,
            min_run=min_run,
            edge_ratio=edge_ratio
        )

        print("\n" + "="*100)
        print(f"TEXT #{i}")
        print("Total runs:", len(runs))

        if not runs:
            continue

        for r_idx, r in enumerate(runs, 1):
            print("\n" + "-"*80)
            print(
                f"RUN #{r_idx} | side={r['side']} | "
                f"block_i={r['block_i']} | "
                f"n_lines={r['n_lines']} | "
                f"span=[{r['run_start']},{r['run_end']})"
            )

            for ln in r["lines"]:
                print("RUN_LINE:", repr(ln))

check_idx = 96789
t1 = df6["text_clean_v3"].iloc[check_idx]

print_short_runs_for_two_texts(t1)                


TEXT #1
Total runs: 44

--------------------------------------------------------------------------------
RUN #1 | side=top30% | block_i=2 | n_lines=2 | span=[76,94)
RUN_LINE: 'عربي'
RUN_LINE: 'Remember Me'

--------------------------------------------------------------------------------
RUN #2 | side=top30% | block_i=3 | n_lines=1 | span=[96,124)
RUN_LINE: 'Forgot Username or Password'

--------------------------------------------------------------------------------
RUN #3 | side=top30% | block_i=5 | n_lines=2 | span=[129,160)
RUN_LINE: 'New Here?'
RUN_LINE: 'Create an account'

--------------------------------------------------------------------------------
RUN #4 | side=top30% | block_i=6 | n_lines=1 | span=[183,187)
RUN_LINE: 'Home'

--------------------------------------------------------------------------------
RUN #5 | side=top30% | block_i=7 | n_lines=1 | span=[189,193)
RUN_LINE: 'News'

--------------------------------------------------------------------------------
RUN #6 | s

In [20]:
def remove_short_line_runs_from_blocks(
    text,
    max_chars: int = 50,
    min_run: int = 1,
    edge_ratio: float = 0.30,
    protected_keywords=PROTECTED_KEYWORDS,
    collapse_blanks: bool = True,
):
    """
    Remove the runs detected by extract_short_line_runs_from_blocks (top/bottom edge only),
    using exact [run_start, run_end) char spans in the normalized text.

    Steps:
      1) normalize text the same way as extract (\\n -> \n, \n{3,}->\n\n)
      2) compute runs (with edge gating + semantic guard)
      3) delete runs by slicing (merge intervals to avoid index shift)
      4) optional: collapse excessive blank lines after deletion
    """
    # --- normalize (MUST match extract) ---
    t = normalize_newlines_loose(text)  

    if not t:
        return t

    runs = extract_short_line_runs_from_blocks(
        t,
        max_chars=max_chars,
        min_run=min_run,
        edge_ratio=edge_ratio,
        protected_keywords=protected_keywords,
    )
    if not runs:
        return t

    # collect & sort intervals
    intervals = sorted([(r["run_start"], r["run_end"]) for r in runs], key=lambda x: x[0])

    # merge overlaps/adjacent (adjacent helps remove contiguous menu chunks cleanly)
    merged = []
    for s, e in intervals:
        if not merged or s > merged[-1][1]:
            merged.append([s, e])
        else:
            merged[-1][1] = max(merged[-1][1], e)

    # delete from left->right by reconstructing
    out_parts = []
    prev = 0
    for s, e in merged:
        out_parts.append(t[prev:s])
        prev = e
    out_parts.append(t[prev:])
    out = "".join(out_parts)

    if collapse_blanks:
        out = out.replace("\r\n", "\n").replace("\r", "\n")
        out = re.sub(r"\n\s*\n+", "\n\n", out)
        out = out.strip()
    else:
        out = out.strip()

    return out

In [21]:
df6["text_clean_v3_"] = df6["text_clean_v3"] .apply(remove_short_line_runs_from_blocks)
df6["text_clean_v4"]  = df6["text_clean_v3_"].apply(remove_short_line_runs_from_blocks) # run twice to catch nested/overlapping runs that may appear after first pass

In [22]:
show_full(df6[["text_clean_v3", "text_clean_v4"]].iloc[check_idx,:].to_frame().T)

text_clean_v3  \
110437  Neural networks, machine learning? Nobel-winning AI science explained\n \n\n \n\nعربي\n \nRemember Me\n\n Forgot Username or Password\n\n \n\n New Here? \n Create an account\n \n \n \n \n \n \n \n \n\n \n \n \nHome\n\nNews\n\nNews by Industry\n\nNews by Region\n\nAmerican\nEurope\nArab World\nAsia\nAfrica\nRSS\n\nPress Distribution\n\nPress releases\nSubmit Your Articles/Press Releases/Reports\nPricing\n\nMarket Data\nEquities Market\n\nGlobal Indices\nMENA Indices\nQutoes & Charts\nEnd Of Day Stocks\nCurrencies\n\nCurrency Convertor\nCross Rates\nHistorical Currencies\nLibor\nMena Stocks\nCommodoties\nOil & Energy\nEconomic Calender\n\nResearch\nPremium Research\n\nFree Research\nCountries\n\nSaudi Arabia\nUAE\nBahrain\nQatar\nKuwait\nJordan\nOman\nEgypt\nLebanon\nIraq\nPalestine\nSyria\nTunisia\nAlgeria\nMorocco\nYemen\n\nSections\n\nEvents\nFinancial Glossary\n\n \n \n \n \n                US     \n      Europe     \n      Arab     \n      Asia     \n      Africa     \n\n|      Politics     \n      Economy     \n Oil&Energy          \n\n                  Entertainment     \n      Sport \n Neural networks, machine learning? Nobel-winning AI science explained\n Date\n10/8/2024 3:23:31 PM\n\n \n\n Share on Facebook\n\n Tweet on Twitter\n\n \n\n \n \n(MENAFN- AFP) The Nobel Prize in Physics was awarded to two scientists on Tuesday for discoveries that laid the groundwork for the artificial intelligence used by hugely popular tools such as ChatGPT.British-Canadian Geoffrey Hinton, known as a "godfather of AI," and US physicist John Hopfield were given the prize for "discoveries and inventions that enable machine learning with artificial neural networks," the Nobel jury said.But what are those, and what does this all mean? Here are some answers.- What are neural networks and machine learning? -Mark van der Wilk, an expert in machine learning at the University of Oxford, told AFP that an artificial neural network is a mathematical construct "loosely inspired" by the human brain. Our brains have a network of cells called neurons, which respond to outside stimuli -- such as things our eyes have seen or ears have heard -- by sending signals to each other.When we learn things, some connections between neurons get stronger, while others get weaker. Unlike traditional computing, which works more like reading a recipe, artificial neural networks roughly mimic this process. The biological neurons are replaced with simple calculations sometimes called "nodes" -- and the incoming stimuli they learn from is replaced by training data.The idea is that this could allow the network to learn over time -- hence the term machine learning. - What did Hopfield discover? -But before machines would be able to learn, another human trait was necessary: memory.Ever struggle to remember a word? Consider the goose. You might cycle through similar words -- goon, good, ghoul -- before striking upon goose. "If you are given a pattern that's not exactly the thing that you need to remember, you need to fill in the blanks," van der Wilk said."That's how you remember a particular memory."This was the idea behind the "Hopfield network" -- also called "associative memory" -- which the physicist developed back in the early 1980s.Hopfield's contribution meant that when an artificial neural network is given something that is slightly wrong, it can cycle through previously stored patterns to find the closest match.This proved a major step forward for AI.- What about Hinton? -In 1985, Hinton revealed his own contribution to the field -- or at least one of them -- called the Boltzmann machine.Named after 19th century physicist Ludwig Boltzmann, the concept introduced an element of randomness.This randomness was ultimately why today's AI-powered image generators can produce endless variations to the same prompt.Hinton also showed that the more layers a network has, "the more complex its behaviour can be".This in turn made it easier to "efficiently 

In [23]:
df7 = df6.copy().drop(columns=['text_v3_length','text_clean_v3_'])
df7['text_v4_length'] = df7['text_clean_v4'].astype(str).apply(len)
df7.columns

Index(['url', 'date', 'language', 'title', 'text', 'text_norm0',
       'text_clean_v1', 'bp_score', 'bp_label', 'text_clean_v2',
       'text_clean_v3', 'text_clean_v4', 'text_v4_length'],
      dtype='object')

In [24]:
df7['text_v4_length'].describe()

count    168539.000000
mean       7423.347480
std        4734.115469
min         356.000000
25%        4570.000000
50%        6451.000000
75%        9027.000000
max      182461.000000
Name: text_v4_length, dtype: float64

#### Language choice/ Country - Specific

In [25]:
def is_country_like_line(s: str) -> bool:
    s = s.strip()
    if not s:
        return False

    if len(s) > 25:
        return False

    if "." in s or "," in s:
        return False

    words = s.split()
    if not (1 <= len(words) <= 3):
        return False

    if all(w[0].isupper() for w in words if w.isalpha()):
        return True
    
    if any(w.lower() in {"is","are","was","were","has","have"} for w in words):
        return False

    return False


def extract_country_runs(text, min_run=5):
    if not text:
        return []

    t = str(text)
    if "\\n" in t and "\n" not in t:
        t = t.replace("\\n", "\n")

    lines = t.splitlines()
    runs = []
    i = 0

    while i < len(lines):
        if is_country_like_line(lines[i]):
            j = i
            while j < len(lines) and is_country_like_line(lines[j]):
                j += 1

            if (j - i) >= min_run:
                runs.append((i, j))  # line indices

            i = j
        else:
            i += 1

    return runs

In [26]:
def print_country_runs(text, min_run=5):
    runs = extract_country_runs(text, min_run=min_run)
    lines = text.splitlines()

    print("Total runs:", len(runs))

    for idx, (start, end) in enumerate(runs, 1):
        print("\n" + "="*80)
        print(f"RUN #{idx} | lines {start}–{end-1} | n_lines={end-start}")
        print("-"*80)

        for i in range(start, end):
            print("LINE:", repr(lines[i]))


print_country_runs(df7["text_clean_v4"].iloc[check_idx])

Total runs: 4

RUN #1 | lines 12–27 | n_lines=16
--------------------------------------------------------------------------------
LINE: 'Saudi Arabia'
LINE: 'UAE'
LINE: 'Bahrain'
LINE: 'Qatar'
LINE: 'Kuwait'
LINE: 'Jordan'
LINE: 'Oman'
LINE: 'Egypt'
LINE: 'Lebanon'
LINE: 'Iraq'
LINE: 'Palestine'
LINE: 'Syria'
LINE: 'Tunisia'
LINE: 'Algeria'
LINE: 'Morocco'
LINE: 'Yemen'

RUN #2 | lines 93–98 | n_lines=6
--------------------------------------------------------------------------------
LINE: 'Privacy Policy'
LINE: 'Contact Us'
LINE: 'Advertise'
LINE: 'About MENAFN'
LINE: 'Jobs'
LINE: 'Feedback'

RUN #3 | lines 105–110 | n_lines=6
--------------------------------------------------------------------------------
LINE: ' Facebook'
LINE: ' Twitter'
LINE: ' Google Plus'
LINE: ' Daily Email'
LINE: ' Linkedin'
LINE: ' RSS'

RUN #4 | lines 117–132 | n_lines=16
--------------------------------------------------------------------------------
LINE: 'Saudi Arabia'
LINE: 'UAE'
LINE: 'Bahrain'
LINE: 'Qa

In [27]:
def remove_country_runs(text, min_run=5, collapse_blanks=True):
    if not text:
        return text

    text = str(text)

    # 统一换行
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    lines = text.splitlines()
    runs = extract_country_runs(text, min_run=min_run)

    if not runs:
        return text.strip()

    # 标记要删除的行
    keep_mask = [True] * len(lines)

    for start, end in runs:
        for i in range(start, end):
            keep_mask[i] = False

    # 重建文本
    cleaned = "\n".join(
        ln for ln, keep in zip(lines, keep_mask) if keep
    )

    if collapse_blanks:
        cleaned = re.sub(r"\n\s*\n+", "\n\n", cleaned)

    return cleaned.strip()

df7["text_clean_v5"] = df7["text_clean_v4"].apply(remove_country_runs)

In [28]:
df8 = df7[['url', 'date', 'language', 'title', 'text','text_clean_v5']].copy().rename(columns={'text_clean_v5': 'text_cleaned'})
df8['text_cleaned_length'] = df8['text_cleaned'].astype(str).apply(len)
df8.columns

Index(['url', 'date', 'language', 'title', 'text', 'text_cleaned',
       'text_cleaned_length'],
      dtype='object')

In [29]:
save_path_v5 = DATA_DIR / "temp" / "text_clean_v5.parquet"
df8.to_parquet(save_path_v5, index=False)

### Keyword pre-filter — Pre-filter docs that likely aren’t about AI impact on industries.

In [61]:
save_path_v5 = DATA_DIR / "temp" / "text_clean_v5.parquet"
df_cleantext = pd.read_parquet(save_path_v5)
df_cleantext.head(2)

,url,date,language,title,text,text_cleaned,text_cleaned_length
0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...",3500
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,\n\nThis AI video of gymnastics might be the f...,This AI video of gymnastics might be the freak...,4409


In [62]:
import re

AI_TERMS = [
    r"artificial intelligence", r"\bai\b", r"\bgenerative ai\b", r"\bgenai\b",
    r"\bllm\b", r"large language model[s]?",
    r"\bmachine learning\b", r"\bml\b", r"\bdeep learning\b",
    r"\bchatgpt\b", r"\bgpt-?\d*\b", r"\bopenai\b", r"\banthropic\b", r"\bclaude\b",
    r"\bgemini\b", r"\bgoogle ai\b", r"\bmicrosoft copilot\b", r"\bcopilot\b"
]

IMPACT_TERMS = [
    r"impact", r"affect", r"effect", r"change", r"shift", r"disrupt",
    r"adopt", r"adoption", r"deploy", r"deployment", r"rollout",
    r"regulat", r"policy", r"law", r"ban", r"risk", r"safety", r"ethic",
    r"job[s]?", r"labor", r"workforce", r"productivit", r"automation",
    r"revenue", r"cost", r"efficien", r"market", r"competition"
]

ai_re = re.compile("(" + "|".join(AI_TERMS) + ")", flags=re.IGNORECASE)
impact_re = re.compile("(" + "|".join(IMPACT_TERMS) + ")", flags=re.IGNORECASE)

In [63]:
# must hit：AI terms & either impact terms in text OR AI terms in title
s = df_cleantext["text_cleaned"].fillna("").astype(str)
mask_ai = s.str.contains(ai_re)
mask_impact = s.str.contains(impact_re)

t = df_cleantext["title"].fillna("").astype(str)
mask_title_ai = t.str.contains(ai_re)

df_ai_impact = df_cleantext[mask_ai & (mask_impact | mask_title_ai)].copy()
print("Total AI-impact articles:", len(df_ai_impact))

C:\Users\ss363\AppData\Local\Temp\ipykernel_5764\129157956.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_ai = s.str.contains(ai_re)
C:\Users\ss363\AppData\Local\Temp\ipykernel_5764\129157956.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_impact = s.str.contains(impact_re)
C:\Users\ss363\AppData\Local\Temp\ipykernel_5764\129157956.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_title_ai = t.str.contains(ai_re)


Total AI-impact articles: 165238


In [64]:
top2_idx = (
    df_ai_impact["text_cleaned_length"]
    .nlargest(2)
    .index
)

df_ai_impact = df_ai_impact.drop(top2_idx)

print(
    "The two longest articles were removed. "
    "The longest contained mostly redundant content, "
    "and the second longest focused on political discussion without meaningful relevance to this project."
)

The two longest articles were removed. The longest contained mostly redundant content, and the second longest focused on political discussion without meaningful relevance to this project.


### Save Final cleaned text

In [76]:
final_text_path = DATA_DIR / "interim" / "df_clean.parquet"
df_ai_impact.to_parquet(final_text_path, index=False)